In [ ]:
# ==========================================================
# STEP 1 : Install Libraries
# ==========================================================

!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install rank-bm25
!pip -q install PyPDF2
!pip -q install python-docx
!pip -q install tqdm
!pip -q install scikit-learn

# ==========================================================
# STEP 2 : Imports
# ==========================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

In [ ]:
# ==========================================================
# STEP 3 : Mount Google Drive
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================================
# STEP 4 : Dataset Path
# ==========================================================

DATA_PATH = "/content/drive/MyDrive/redrob"

print(os.listdir(DATA_PATH))

In [ ]:
# ==========================================================
# STEP 5 : File Paths
# ==========================================================

candidate_file = os.path.join(DATA_PATH,"candidates.jsonl")
schema_file = os.path.join(DATA_PATH,"candidate_schema.json")
job_file = os.path.join(DATA_PATH,"job_description.pdf")
signal_file = os.path.join(DATA_PATH,"redrob_signals_document.pdf")
submission_file = os.path.join(DATA_PATH,"submission.csv")

In [ ]:
# ==========================================================
# STEP 6 : Load Candidates
# ==========================================================

candidates = []

with open(candidate_file,"r") as f:

    for line in f:

        candidates.append(json.loads(line))

print("Candidates Loaded :",len(candidates))

In [ ]:
# ==========================================================
# STEP 7 : AI Skill Weights
# ==========================================================

SKILL_WEIGHTS = {

    # Core Retrieval

    "embeddings":12,
    "retrieval":12,
    "ranking":12,

    "vector database":11,

    "pinecone":10,
    "weaviate":10,
    "milvus":10,
    "faiss":10,
    "qdrant":10,

    "elasticsearch":9,
    "opensearch":9,

    # Production NLP

    "sentence transformers":9,
    "recommendation systems":9,
    "information retrieval":9,

    # Python

    "python":8,

    # LLM

    "llms":7,
    "fine-tuning llms":7,
    "lora":6,
    "qlora":6,
    "peft":6,

    # Supporting

    "transformers":5,
    "hugging face transformers":5,
    "huggingface":5,

    "weights & biases":4,

    "nlp":4,

    # VERY LOW

    "langchain":2,
    "prompt engineering":2,
    "rag":2
}

In [ ]:
# ==========================================================
# STEP 8 : AI Skill Weights
# ==========================================================
AI_TITLES = [

    "machine learning engineer",
    "ml engineer",
    "ai engineer",
    "applied scientist",
    "research scientist",
    "nlp engineer",
    "llm engineer",
    "search engineer",
    "recommendation engineer",
    "data scientist"
]

In [ ]:
# ==========================================================
# STEP 9 : AI Skill Weights
# ==========================================================
PRODUCT_COMPANIES = {

"Google",
"Meta",
"Microsoft",
"Apple",
"Amazon",
"Netflix",
"Uber",
"Flipkart",
"PhonePe",
"Paytm",
"Razorpay",
"Swiggy",
"Zomato",
"Ola",
"Meesho",
"Yellow.ai",
"Sarvam AI",
"OpenAI",
"Anthropic",
"Niramai",
"Salesforce",
"Locobuzz",
"Mad Street Den"

}

In [ ]:
# ==========================================================
# STEP 10 : AI Skill Weights
# ==========================================================
CONSULTING = {

"TCS",
"Infosys",
"Wipro",
"Accenture",
"Cognizant",
"Capgemini",
"HCL",
"Mindtree",
"LTIMindtree"

}

In [ ]:
# ==========================================================
# STEP 11 : AI Skill Weights
# ==========================================================
PRODUCTION_WORDS = [

"production",
"deployed",
"deployment",

"retrieval",
"ranking",
"recommendation",

"embedding",
"embeddings",

"vector",

"search",

"evaluation",

"ndcg",
"mrr",
"map",

"a/b",

"latency",

"pipeline",

"real users",

"millions",

"faiss",

"pinecone",

"milvus",

"weaviate"

]

In [ ]:
# ==========================================================
# STEP 12 : AI Skill Weights
# ==========================================================
def experience_score(exp):

    if 5 <= exp <= 9:
        return 1.0

    elif 4 <= exp < 5:
        return 0.85

    elif 9 < exp <= 10:
        return 0.85

    elif 3 <= exp < 4:
        return 0.55

    elif 10 < exp <= 12:
        return 0.45

    else:
        return 0.15

In [ ]:
# ==========================================================
# STEP 13 : Weighted AI Skill Score
# ==========================================================

def weighted_ai_score(skills):

    score = 0
    matched = []

    for skill in skills:

        name = skill["name"].lower()

        if name in SKILL_WEIGHTS:

            score += SKILL_WEIGHTS[name]
            matched.append(name)

    max_score = sum(SKILL_WEIGHTS.values())

    normalized = score / max_score

    return normalized, matched

In [ ]:
# ==========================================================
# STEP 14 : AI Career Score
# ==========================================================

def ai_career_score(candidate):

    score = 0

    profile = candidate["profile"]

    title = profile["current_title"].lower()

    summary = profile["summary"].lower()

    if any(t in title for t in AI_TITLES):
        score += 4

    if "machine learning" in summary:
        score += 2

    if "retrieval" in summary:
        score += 2

    if "ranking" in summary:
        score += 2

    if "recommendation" in summary:
        score += 2

    if "embeddings" in summary:
        score += 2

    return min(score/14,1)

In [ ]:
# ==========================================================
# STEP 15 : Production ML Experience
# ==========================================================

def production_ml_score(candidate):

    total = 0

    for job in candidate["career_history"]:

        desc = job["description"].lower()

        for word in PRODUCTION_WORDS:

            if word in desc:
                total += 1

    return min(total/20,1)

In [ ]:
# ==========================================================
# STEP 16 : Product Company Score
# ==========================================================

def product_company_score(candidate):

    score = 0

    for job in candidate["career_history"]:

        if job["company"] in PRODUCT_COMPANIES:
            score += 1

    return min(score/4,1)

In [ ]:
# ==========================================================
# STEP 17 : Consulting Penalty
# ==========================================================

def consulting_penalty(candidate):

    consulting = 0

    for job in candidate["career_history"]:

        if job["company"] in CONSULTING:
            consulting += 1

    if consulting == len(candidate["career_history"]):
        return 1

    return 0

In [ ]:
# ==========================================================
# STEP 18 : Fake AI Detection
# ==========================================================

def fake_ai_penalty(candidate):

    title = candidate["profile"]["current_title"].lower()

    ai_titles = any(x in title for x in AI_TITLES)

    weighted_score, _ = weighted_ai_score(candidate["skills"])

    production = production_ml_score(candidate)

    # Lots of AI skills but no AI career
    if (not ai_titles) and weighted_score > 0.40:
        return 0.30

    # AI skills but no production work
    if production < 0.20 and weighted_score > 0.40:
        return 0.20

    return 0

In [ ]:
# ==========================================================
# STEP 19 : Behaviour Score
# ==========================================================

def behaviour_score(candidate):

    s = candidate["redrob_signals"]

    score = 0

    if s["open_to_work_flag"]:
        score += 2

    score += s["recruiter_response_rate"]*2

    score += s["interview_completion_rate"]*2

    score += min(s["github_activity_score"],100)/100

    if s["willing_to_relocate"]:
        score += 1

    if s["notice_period_days"] <= 30:
        score += 1

    score += min(s["saved_by_recruiters_30d"]/40,1)

    score += min(s["search_appearance_30d"]/500,1)

    return score/11

In [ ]:
# ==========================================================
# STEP 20 : Extract Features
# ==========================================================

def extract_features(candidate):

    weighted_score,_ = weighted_ai_score(candidate["skills"])

    return {

        "candidate_id":candidate["candidate_id"],

        "experience_years":
        candidate["profile"]["years_of_experience"],

        "experience_score":
        experience_score(candidate["profile"]["years_of_experience"]),

        "weighted_ai_score":
        weighted_score,

        "ai_career_score":
        ai_career_score(candidate),

        "production_score":
        production_ml_score(candidate),

        "product_company_score":
        product_company_score(candidate),

        "consulting_penalty":
        consulting_penalty(candidate),

        "behaviour_score":
        behaviour_score(candidate),

        "fake_ai_penalty":
        fake_ai_penalty(candidate)

    }

In [ ]:
# ==========================================================
# STEP 21 : Candidate Text Builder
# ==========================================================

def build_candidate_text(candidate):

    profile = candidate["profile"]

    text = ""

    # Current profile
    text += profile["current_title"] + " "
    text += profile["headline"] + " "
    text += profile["summary"] + " "

    # Career history (most important)
    for job in candidate["career_history"]:

        text += job["title"] + " "
        text += job["company"] + " "
        text += job["description"] + " "

    # Skills (least important)
    for skill in candidate["skills"]:

        text += skill["name"] + " "

    return text

In [ ]:
# ==========================================================
# STEP 22 : Build Candidate Corpus
# ==========================================================

candidate_texts = []

for candidate in tqdm(candidates):

    candidate_texts.append(
        build_candidate_text(candidate)
    )

print("Candidate Texts :", len(candidate_texts))

In [ ]:
import docx

# ==========================================================
# STEP 23 : Load Job Description & Generate Embedding
# ==========================================================

def extract_text_from_docx(docx_path):
    doc = docx.Document(docx_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    return '\n'.join(full_text)

# Update job_file to point to the correct .docx file
job_file = os.path.join(DATA_PATH,"job_description (1).docx")

job_text = extract_text_from_docx(job_file)

model = SentenceTransformer('all-MiniLM-L6-v2')

job_embedding = model.encode(

    job_text,

    normalize_embeddings=True
)

In [ ]:
# ==========================================================
# STEP 24 : Load Saved Features & Embeddings
# ==========================================================

DATA_PATH = "/content/drive/MyDrive/redrob"

features_df = pd.read_pickle(DATA_PATH + "/features_backup.pkl")

candidate_embeddings = np.load(DATA_PATH + "/candidate_embeddings.npy")

job_embedding = np.load(DATA_PATH + "/job_embedding.npy")

print("Loaded successfully!")
print(features_df.shape)
print(candidate_embeddings.shape)
print(job_embedding.shape)

In [ ]:
# ==========================================================
# STEP 25 : Semantic Scores
# ==========================================================

semantic_scores = np.dot(
    candidate_embeddings,
    job_embedding
)

features_df["semantic_score"] = semantic_scores

features_df.head()

In [ ]:
# ==========================================
# STEP 26 : Rebuild Feature Table
# ==========================================

feature_rows = []

for candidate in tqdm(candidates):
    feature_rows.append(
        extract_features(candidate)
    )

features_df = pd.DataFrame(feature_rows)

print(features_df.head())

In [ ]:
# ==========================================
# STEP 27 : Add Semantic Scores
# ==========================================

semantic_scores = np.dot(
    candidate_embeddings,
    job_embedding
)

features_df["semantic_score"] = semantic_scores

features_df.head()

In [ ]:
# STEP 28
AI_TITLES = [

    "ai engineer",
    "machine learning engineer",
    "ml engineer",
    "applied scientist",
    "nlp engineer",
    "research engineer",
    "recommendation engineer",
    "retrieval engineer",
    "search engineer",
    "llm engineer",
    "data scientist"

]

def ai_title_score(title):

    title = title.lower()

    for t in AI_TITLES:
        if t in title:
            return 1

    return 0

In [ ]:
# STEP 29
title_scores = []

for c in candidates:

    title_scores.append(
        ai_title_score(
            c["profile"]["current_title"]
        )
    )

features_df["title_score"] = title_scores

In [ ]:
# STEP 30
features_df["production_score"] = (
    features_df["production_score"] /
    features_df["production_score"].max()
)

features_df["product_company_score"] = (
    features_df["product_company_score"] /
    features_df["product_company_score"].max()
)

In [ ]:
# STEP 31
features_df["experience_score"] = (
    features_df["experience_years"]
    .apply(experience_score)
)

In [ ]:
# STEP 32
features_df["final_score"] = (

    0.22 * features_df["semantic_score"]

    + 0.20 * features_df["ai_career_score"]

    + 0.15 * features_df["weighted_ai_score"]

    + 0.12 * features_df["experience_score"]

    + 0.10 * features_df["production_score"] # Corrected column name

    + 0.06 * features_df["title_score"]

    + 0.05 * features_df["product_company_score"] # Corrected column name

    + 0.10 * features_df["behaviour_score"] # Added behaviour_score

    - 0.10 * features_df["fake_ai_penalty"] # Adding penalty

    - 0.05 * features_df["consulting_penalty"] # Adding penalty

)

In [ ]:
# STEP 33
features_df = features_df.sort_values(
    "final_score",
    ascending=False
)

features_df["rank"] = range(
    1,
    len(features_df) + 1
)

In [ ]:
# ==========================================================
# STEP 34 : Recruiter Reasoning
# ==========================================================

def generate_reason(row):

    reasons = []

    if row.semantic_score > 0.70:
        reasons.append("Strong semantic match")

    if row.ai_career_score > 0.60:
        reasons.append("Strong AI career")

    if row.production_score > 0.50:
        reasons.append("Production ML experience")

    if row.weighted_ai_score > 0.50:
        reasons.append("Relevant AI skills")

    if row.product_company_score > 0:
        reasons.append("Product company experience")

    if row.behaviour_score > 0.60:
        reasons.append("Strong recruiter signals")

    if row.experience_score == 1:
        reasons.append("Ideal experience")

    if row.fake_ai_penalty > 0:
        reasons.append("AI keyword inflation detected")

    if row.consulting_penalty > 0:
        reasons.append("Consulting-only background")

    return ", ".join(reasons)


features_df["reasoning"] = features_df.apply(
    generate_reason,
    axis=1
)

In [ ]:
# ==========================================================
# STEP 35 : Top Candidates
# ==========================================================

features_df.head(20)

In [ ]:
# ==========================================================
# STEP 36 : Generate System Architecture Diagram
# ==========================================================

!apt-get -qq install graphviz
!pip -q install graphviz

from graphviz import Digraph

dot = Digraph(
    "RecruitAI",
    format="png"
)

dot.attr(rankdir="LR")
dot.attr(fontname="Helvetica")

dot.attr("node",
         shape="box",
         style="rounded,filled",
         color="lightblue",
         fontname="Helvetica")

# Input
dot.node("A", "Job Description")

# Embedding
dot.node("B", "Sentence Transformer\n(all-MiniLM-L6-v2)")

# Candidate Data
dot.node("C", "Candidate Profiles")
dot.node("D", "Career History")
dot.node("E", "Skills")
dot.node("F", "Redrob Signals")

# Feature Engineering
dot.node("G", "Feature Engineering")

# Semantic Matching
dot.node("H", "Semantic Similarity")

# Ranking
dot.node("I","RecruitAI Scoring Engine")

# Explainability
dot.node("J", "Reasoning Generator")

# Output
dot.node("K", "Ranked Candidate Shortlist")

# Connections
dot.edge("A","B")

dot.edge("C","G")
dot.edge("D","G")
dot.edge("E","G")
dot.edge("F","G")

dot.edge("B","H")
dot.edge("G","I")
dot.edge("H","I")

dot.edge("I","J")
dot.edge("J","K")

dot.render("RecruitAI_Architecture", cleanup=True)

print("Architecture diagram saved successfully.")

In [ ]:
from IPython.display import Image

Image("RecruitAI_Architecture.png")